# TAAC2026 Dataset EDA Report

腾讯广告算法大赛 — Post-Click Conversion Rate (PCVR) Prediction

Data source: `hf://datasets/TAAC2026/data_sample_1000/demo_1000.parquet` (1000 samples)

## 关键发现速览

| 维度 | 发现 |
|------|------|
| **数据量** | 1000 样本，120 列，时间窗口 781s（约 13 分钟） |
| **标签分布** | 点击 87.6%，转化 12.4%，类别不平衡 ~1:7.1 |
| **转化延迟** | 中位数 192s，均值 229s，范围 2~832s |
| **用户特征** | 46 个整数特征（28 标量 + 10 列表），fid 101 缺失率 91%，fid 62~66 与 fid 89~91 分别对应同一底层信号的离散/稠密两部分 |
| **物品特征** | 14 个整数特征（13 标量 + 1 列表），fid 83/84/85 缺失率 83.2%（高度稀疏信号），无 item_dense 列 |
| **密集特征** | 10 个用户向量：fid 61 为 256 维 embedding，fid 87 为 320 维 embedding，fid 62~66/89~91 为变长向量 |
| **序列特征** | 4 个 domain（a/b/c/d）共 45 列，domain_a 最长（中位 582）、缺失率 0.5%，domain_d 缺失率最高 8% |
| **缺失模式** | 用户标量特征平均缺失 31.7%，物品列表 43.9%，序列仅 2.3%；多数高缺失特征在正负样本间缺失率接近，无明显 label 偏向 |
| **曝光特征** | 1000 个独立用户（无重复曝光），837 个独立物品，部分物品最高曝光 17 次 |

**数据质量注意**：
- `user_int_feats_99` ~ `user_int_feats_103` 缺失率 >80%，信号极稀疏
- `item_int_feats_83/84/85` 同步缺失 83.2%，可能为同一上游信号的不同侧面
- 序列中 fid 39/67/74/19 等列值域为 timestamp 量级，为各 domain 的时间戳列
- 原始数据中 -1 和 0 均被视为 padding

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_parquet("hf://datasets/TAAC2026/data_sample_1000/demo_1000.parquet")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

/Users/lvxintao/miniconda3/envs/agent/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shape: 1000 rows x 120 columns
Memory: 7852.3 KB


## 1. Column Classification

In [2]:
# Classify columns by prefix
meta_cols = ['user_id', 'item_id', 'label_type', 'label_time', 'timestamp']
user_int_cols = [c for c in df.columns if c.startswith('user_int_feats_')]
item_int_cols = [c for c in df.columns if c.startswith('item_int_feats_')]
user_dense_cols = [c for c in df.columns if c.startswith('user_dense_feats_')]
item_dense_cols = [c for c in df.columns if c.startswith('item_dense_feats_')]
seq_cols = [c for c in df.columns if 'seq_' in c and 'domain' in c]

col_groups = {
    'Meta': meta_cols,
    'User Int': user_int_cols,
    'Item Int': item_int_cols,
    'User Dense': user_dense_cols,
    'Item Dense': item_dense_cols,
    'Sequences': seq_cols,
}

print("Column Groups:")
for name, cols in col_groups.items():
    print(f"  {name:15s}: {len(cols):3d} columns")
print(f"  {'Total':15s}: {sum(len(v) for v in col_groups.values()):3d} columns")

Column Groups:
  Meta           :   5 columns
  User Int       :  46 columns
  Item Int       :  14 columns
  User Dense     :  10 columns
  Item Dense     :   0 columns
  Sequences      :  45 columns
  Total          : 120 columns


## 2. Label Analysis

In [3]:
print("=== label_type Distribution ===")
label_counts = df['label_type'].value_counts().sort_index()
for val, cnt in label_counts.items():
    print(f"  label_type={val}: {cnt} ({cnt/len(df)*100:.1f}%)")

positive_rate = (df['label_type'] == 2).mean()
print(f"\nPositive rate (label_type==2): {positive_rate:.4f} ({positive_rate*100:.2f}%)")
print(f"Class imbalance ratio: 1:{(1-positive_rate)/positive_rate:.1f}")

=== label_type Distribution ===
  label_type=1: 876 (87.6%)
  label_type=2: 124 (12.4%)

Positive rate (label_type==2): 0.1240 (12.40%)
Class imbalance ratio: 1:7.1


In [4]:
fig, ax = plt.subplots(figsize=(6, 4))
labels = ['Click (1)', 'Conversion (2)']
counts = [int((df['label_type'] == 1).sum()), int((df['label_type'] == 2).sum())]
colors = ['#4C72B0', '#DD8452']
bars = ax.bar(labels, counts, color=colors, width=0.5)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            f'{cnt} ({cnt/len(df)*100:.1f}%)', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Count')
ax.set_title('Label Distribution')
ax.set_ylim(0, max(counts) * 1.2)
plt.tight_layout()
plt.savefig('eda/figures/eda_label_dist.png', dpi=150)
plt.close()

## 3. Meta Features

In [5]:
print("=== Meta Features ===")
print(f"user_id:  {df['user_id'].nunique()} unique, min={df['user_id'].min()}, max={df['user_id'].max()}")
print(f"item_id:  {df['item_id'].nunique()} unique, min={df['item_id'].min()}, max={df['item_id'].max()}")
print(f"\nUser repeat rate: {len(df) - df['user_id'].nunique()} repeated exposures out of {len(df)}")
print(f"Item repeat rate: {len(df) - df['item_id'].nunique()} repeated exposures out of {len(df)}")

# User exposure frequency
user_exp_freq = df['user_id'].value_counts()
print(f"\nUser exposure frequency:")
print(f"  mean={user_exp_freq.mean():.2f}, median={user_exp_freq.median()}, "
      f"max={user_exp_freq.max()}, min={user_exp_freq.min()}")

item_exp_freq = df['item_id'].value_counts()
print(f"\nItem exposure frequency:")
print(f"  mean={item_exp_freq.mean():.2f}, median={item_exp_freq.median()}, "
      f"max={item_exp_freq.max()}, min={item_exp_freq.min()}")

=== Meta Features ===
user_id:  1000 unique, min=2727076, max=12728427
item_id:  837 unique, min=6854, max=278202253

User repeat rate: 0 repeated exposures out of 1000
Item repeat rate: 163 repeated exposures out of 1000

User exposure frequency:
  mean=1.00, median=1.0, max=1, min=1

Item exposure frequency:
  mean=1.19, median=1.0, max=17, min=1


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(user_exp_freq, bins=20, color='#4C72B0', edgecolor='white')
axes[0].set_xlabel('Exposures per User')
axes[0].set_ylabel('Count')
axes[0].set_title('User Exposure Frequency')

axes[1].hist(item_exp_freq, bins=20, color='#DD8452', edgecolor='white')
axes[1].set_xlabel('Exposures per Item')
axes[1].set_ylabel('Count')
axes[1].set_title('Item Exposure Frequency')
plt.tight_layout()
plt.savefig('eda/figures/eda_exposure_freq.png', dpi=150)
plt.close()

In [7]:
# Timestamp analysis
print("=== Timestamp Analysis ===")
ts_min = df['timestamp'].min()
ts_max = df['timestamp'].max()
duration_sec = ts_max - ts_min
print(f"timestamp range: {ts_min} - {ts_max} (duration: {duration_sec}s = {duration_sec/60:.1f}min)")
print(f"  -> {datetime.fromtimestamp(ts_min)} to {datetime.fromtimestamp(ts_max)}")

delta = df['label_time'] - df['timestamp']
print(f"\nlabel_time - timestamp (delay):")
print(f"  mean={delta.mean():.1f}s, median={delta.median():.1f}s, "
      f"min={delta.min()}s, max={delta.max()}s")
print(f"  std={delta.std():.1f}s")

=== Timestamp Analysis ===
timestamp range: 1772725000 - 1772725781 (duration: 781s = 13.0min)
  -> 2026-03-05 23:36:40 to 2026-03-05 23:49:41

label_time - timestamp (delay):
  mean=228.5s, median=191.5s, min=2s, max=832s
  std=171.7s


In [8]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(delta, bins=50, color='#55A868', edgecolor='white')
ax.set_xlabel('Delay (seconds)')
ax.set_ylabel('Count')
ax.set_title('Label Time vs Exposure Time Delay')
ax.axvline(delta.median(), color='red', linestyle='--', label=f'Median={delta.median():.0f}s')
ax.axvline(delta.mean(), color='orange', linestyle='--', label=f'Mean={delta.mean():.0f}s')
ax.legend()
plt.tight_layout()
plt.savefig('eda/figures/eda_delay_dist.png', dpi=150)
plt.close()

## 4. User Integer Features

In [9]:
# Scalar vs list classification
def get_col_type(series):
    non_null = series.dropna()
    if len(non_null) == 0:
        return 'all_null'
    first = non_null.iloc[0]
    if isinstance(first, (list, np.ndarray)):
        return 'list'
    return 'scalar'

user_int_summary = []
for c in user_int_cols:
    ctype = get_col_type(df[c])
    nulls = df[c].isna().sum()
    entry = {'col': c, 'type': ctype, 'nulls': nulls, 'null_pct': nulls/len(df)*100}
    if ctype == 'scalar':
        entry['nunique'] = df[c].nunique()
    elif ctype == 'list':
        lens = df[c].dropna().apply(len)
        entry['len_mean'] = lens.mean()
        entry['len_min'] = lens.min()
        entry['len_max'] = lens.max()
    user_int_summary.append(entry)

ui_df = pd.DataFrame(user_int_summary)
print(ui_df.to_string(index=False))

               col   type  nulls  null_pct  nunique  len_mean  len_min  len_max
  user_int_feats_1 scalar      0       0.0      3.0       NaN      NaN      NaN
  user_int_feats_3 scalar     30       3.0    341.0       NaN      NaN      NaN
  user_int_feats_4 scalar     30       3.0    268.0       NaN      NaN      NaN
 user_int_feats_15   list    139      13.9      NaN  3.753775      1.0     13.0
 user_int_feats_48 scalar      2       0.2     52.0       NaN      NaN      NaN
 user_int_feats_49 scalar      7       0.7      2.0       NaN      NaN      NaN
 user_int_feats_50 scalar      4       0.4      2.0       NaN      NaN      NaN
 user_int_feats_51 scalar      1       0.1      5.0       NaN      NaN      NaN
 user_int_feats_52 scalar      1       0.1     36.0       NaN      NaN      NaN
 user_int_feats_53 scalar      1       0.1    264.0       NaN      NaN      NaN
 user_int_feats_54 scalar    368      36.8    462.0       NaN      NaN      NaN
 user_int_feats_55 scalar     19       1

In [10]:
# Visualize: null rate + cardinality for scalar user_int features
scalar_ui = ui_df[ui_df['type'] == 'scalar'].copy()
scalar_ui['fid'] = scalar_ui['col'].apply(lambda x: int(x.split('_')[-1]))
scalar_ui = scalar_ui.sort_values('fid')

fig, ax1 = plt.subplots(figsize=(14, 5))
x = range(len(scalar_ui))
bars = ax1.bar(x, scalar_ui['null_pct'], color='#C44E52', alpha=0.7, label='Null %')
ax1.set_ylabel('Null Rate (%)', color='#C44E52')
ax1.set_xlabel('Feature ID')
ax1.set_xticks(x)
ax1.set_xticklabels(scalar_ui['fid'].astype(str), rotation=90, fontsize=7)

ax2 = ax1.twinx()
ax2.plot(x, scalar_ui['nunique'], color='#4C72B0', marker='o', markersize=3, label='N-unique')
ax2.set_ylabel('Unique Values', color='#4C72B0')
ax2.set_title('User Integer Features: Null Rate & Cardinality')
fig.tight_layout()
plt.savefig('eda/figures/eda_user_int_overview.png', dpi=150)
plt.close()

In [11]:
# Visualize: list features length distribution
list_ui = ui_df[ui_df['type'] == 'list'].copy()
list_ui['fid'] = list_ui['col'].apply(lambda x: int(x.split('_')[-1]))
list_ui = list_ui.sort_values('fid')

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(list_ui))
ax.bar(x, list_ui['len_mean'], color='#8172B3', alpha=0.7)
ax.errorbar(x, list_ui['len_mean'],
            yerr=[list_ui['len_mean'] - list_ui['len_min'], list_ui['len_max'] - list_ui['len_mean']],
            fmt='none', color='black', capsize=3, alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(list_ui['fid'].astype(str), rotation=90)
ax.set_ylabel('Mean Length')
ax.set_xlabel('Feature ID')
ax.set_title('User Integer List Features: Length Statistics')
fig.tight_layout()
plt.savefig('eda/figures/eda_user_int_list.png', dpi=150)
plt.close()

## 5. Item Integer Features

In [12]:
item_int_summary = []
for c in item_int_cols:
    ctype = get_col_type(df[c])
    nulls = df[c].isna().sum()
    entry = {'col': c, 'type': ctype, 'nulls': nulls, 'null_pct': nulls/len(df)*100}
    if ctype == 'scalar':
        entry['nunique'] = df[c].nunique()
    elif ctype == 'list':
        lens = df[c].dropna().apply(len)
        entry['len_mean'] = lens.mean()
        entry['len_min'] = lens.min()
        entry['len_max'] = lens.max()
    item_int_summary.append(entry)

ii_df = pd.DataFrame(item_int_summary)
print(ii_df.to_string(index=False))

              col   type  nulls  null_pct  nunique  len_mean  len_min  len_max
 item_int_feats_5 scalar      2       0.2     82.0       NaN      NaN      NaN
 item_int_feats_6 scalar      2       0.2    216.0       NaN      NaN      NaN
 item_int_feats_7 scalar      2       0.2    349.0       NaN      NaN      NaN
 item_int_feats_8 scalar      2       0.2    226.0       NaN      NaN      NaN
 item_int_feats_9 scalar      2       0.2     24.0       NaN      NaN      NaN
item_int_feats_10 scalar      2       0.2    110.0       NaN      NaN      NaN
item_int_feats_11   list    439      43.9      NaN   3.71836      1.0     20.0
item_int_feats_12 scalar      2       0.2    352.0       NaN      NaN      NaN
item_int_feats_13 scalar      2       0.2      8.0       NaN      NaN      NaN
item_int_feats_16 scalar      2       0.2    662.0       NaN      NaN      NaN
item_int_feats_81 scalar      2       0.2      3.0       NaN      NaN      NaN
item_int_feats_83 scalar    832      83.2     22.0  

In [13]:
scalar_ii = ii_df[ii_df['type'] == 'scalar'].copy()
scalar_ii['fid'] = scalar_ii['col'].apply(lambda x: int(x.split('_')[-1]))
scalar_ii = scalar_ii.sort_values('fid')

fig, ax1 = plt.subplots(figsize=(10, 4))
x = range(len(scalar_ii))
ax1.bar(x, scalar_ii['null_pct'], color='#C44E52', alpha=0.7)
ax1.set_ylabel('Null Rate (%)', color='#C44E52')
ax1.set_xticks(x)
ax1.set_xticklabels(scalar_ii['fid'].astype(str))
ax1.set_title('Item Integer Features: Null Rate & Cardinality')

ax2 = ax1.twinx()
ax2.plot(x, scalar_ii['nunique'], color='#4C72B0', marker='o', label='N-unique')
ax2.set_ylabel('Unique Values', color='#4C72B0')
ax2.legend(loc='upper right')
fig.tight_layout()
plt.savefig('eda/figures/eda_item_int_overview.png', dpi=150)
plt.close()

## 6. User Dense Features

In [14]:
user_dense_summary = []
for c in user_dense_cols:
    ctype = get_col_type(df[c])
    nulls = df[c].isna().sum()
    entry = {'col': c, 'type': ctype, 'nulls': nulls, 'null_pct': nulls/len(df)*100}
    if ctype == 'list':
        lens = df[c].dropna().apply(len)
        entry['len_mean'] = lens.mean()
        entry['len_min'] = lens.min()
        entry['len_max'] = lens.max()
        # Compute element-level stats by flattening
        all_vals = [v for lst in df[c].dropna() for v in lst]
        if all_vals:
            entry['val_mean'] = np.mean(all_vals)
            entry['val_std'] = np.std(all_vals)
            entry['val_min'] = np.min(all_vals)
            entry['val_max'] = np.max(all_vals)
    user_dense_summary.append(entry)

ud_df = pd.DataFrame(user_dense_summary)
print(ud_df.to_string(index=False))

                col type  nulls  null_pct   len_mean  len_min  len_max      val_mean       val_std  val_min      val_max
user_dense_feats_61 list      2       0.2 256.000000      256      256 -1.122495e-03      0.062344 -0.25107 2.012991e-01
user_dense_feats_62 list     70       7.0   2.273118        1        5  1.364007e+05 342585.625000 32.00000 4.954893e+06
user_dense_feats_63 list     70       7.0   2.747312        1       11  1.287053e+05 333811.843750  6.00000 4.860850e+06
user_dense_feats_64 list     70       7.0   4.109677        1       18  1.217089e+05 410811.156250 23.00000 1.276234e+07
user_dense_feats_65 list     80       8.0   6.209783        1       49  1.468817e+05 535104.812500  5.00000 1.840111e+07
user_dense_feats_66 list     86       8.6   7.810722        1       66  1.989356e+05 717726.750000  5.00000 1.840111e+07
user_dense_feats_87 list     15       1.5 320.000000      320      320 -4.027122e-03      0.109841 -0.68420 6.844000e-01
user_dense_feats_89 list     55 

In [15]:
fig, ax = plt.subplots(figsize=(10, 4))
ud_sorted = ud_df.sort_values('col').copy()
ud_sorted['fid'] = ud_sorted['col'].apply(lambda x: int(x.split('_')[-1]))
x = range(len(ud_sorted))
ax.bar(x, ud_sorted['len_mean'], color='#4C72B0', alpha=0.7)
ax.errorbar(x, ud_sorted['len_mean'],
            yerr=[ud_sorted['len_mean'] - ud_sorted['len_min'],
                  ud_sorted['len_max'] - ud_sorted['len_mean']],
            fmt='none', color='black', capsize=3, alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(ud_sorted['fid'].astype(str), rotation=90)
ax.set_ylabel('Mean Length')
ax.set_title('User Dense Features: Vector Length Statistics')
fig.tight_layout()
plt.savefig('eda/figures/eda_user_dense_len.png', dpi=150)
plt.close()

## 7. Sequence Features

In [16]:
# Group sequence columns by domain
seq_domains = {}
for c in seq_cols:
    # domain_a_seq_38 -> domain=a
    parts = c.split('_')
    domain = parts[1]  # a, b, c, d
    seq_domains.setdefault(domain, []).append(c)

for d in sorted(seq_domains.keys()):
    seq_domains[d] = sorted(seq_domains[d])

print("Sequence Domains:")
for d, cols in sorted(seq_domains.items()):
    fids = [int(c.split('_')[-1]) for c in cols]
    print(f"  domain_{d}: {len(cols)} features, fids={sorted(fids)}")

Sequence Domains:
  domain_a: 9 features, fids=[38, 39, 40, 41, 42, 43, 44, 45, 46]
  domain_b: 14 features, fids=[67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 88]
  domain_c: 12 features, fids=[27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 47]
  domain_d: 10 features, fids=[17, 18, 19, 20, 21, 22, 23, 24, 25, 26]


In [17]:
# Per-domain: null rate, sequence length stats
seq_summary = []
for d, cols in sorted(seq_domains.items()):
    for c in cols:
        fid = int(c.split('_')[-1])
        nulls = df[c].isna().sum()
        non_null = df[c].dropna()
        entry = {'domain': d, 'fid': fid, 'col': c, 'nulls': nulls, 'null_pct': nulls/len(df)*100}
        if len(non_null) > 0:
            lens = non_null.apply(len)
            entry['len_mean'] = lens.mean()
            entry['len_median'] = lens.median()
            entry['len_min'] = lens.min()
            entry['len_max'] = lens.max()
            # Value range
            all_vals = [v for lst in non_null for v in lst]
            entry['val_min'] = min(all_vals)
            entry['val_max'] = max(all_vals)
            entry['val_nunique'] = len(set(all_vals))
        seq_summary.append(entry)

seq_df = pd.DataFrame(seq_summary)
print(seq_df.to_string(index=False))

domain  fid             col  nulls  null_pct    len_mean  len_median  len_min  len_max    val_min    val_max  val_nunique
     a   38 domain_a_seq_38      5       0.5  704.609045       582.0        1     1888         -1    1201292        17776
     a   39 domain_a_seq_39      5       0.5  704.609045       582.0        1     1888 1712419199 1772725487       321733
     a   40 domain_a_seq_40      5       0.5  704.609045       582.0        1     1888          0         18           17
     a   41 domain_a_seq_41      5       0.5  704.609045       582.0        1     1888          2         11            9
     a   42 domain_a_seq_42      5       0.5  704.609045       582.0        1     1888          0       1017          407
     a   43 domain_a_seq_43      5       0.5  704.609045       582.0        1     1888          0       3449         1263
     a   44 domain_a_seq_44      5       0.5  704.609045       582.0        1     1888          0      15146         3002
     a   45 domain_a_seq

In [18]:
# Visualize: per-domain null rate comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for idx, (d, cols) in enumerate(sorted(seq_domains.items())):
    ax = axes[idx // 2, idx % 2]
    domain_data = seq_df[seq_df['domain'] == d].copy()
    domain_data = domain_data.sort_values('fid')
    x = range(len(domain_data))
    ax.bar(x, domain_data['null_pct'], color='#C44E52', alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(domain_data['fid'].astype(str), fontsize=7)
    ax.set_title(f'Domain {d} — Null Rate')
    ax.set_ylabel('Null %')
fig.suptitle('Sequence Features: Null Rate by Domain', fontsize=14)
fig.tight_layout()
plt.savefig('eda/figures/eda_seq_null_by_domain.png', dpi=150)
plt.close()

In [19]:
# Visualize: per-domain sequence length distribution (first feature per domain as representative)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for idx, (d, cols) in enumerate(sorted(seq_domains.items())):
    ax = axes[idx // 2, idx % 2]
    rep_col = cols[0]
    non_null = df[rep_col].dropna()
    if len(non_null) > 0:
        lens = non_null.apply(len)
        ax.hist(lens, bins=50, color='#4C72B0', edgecolor='white')
        ax.axvline(lens.median(), color='red', linestyle='--', label=f'Median={lens.median():.0f}')
        ax.axvline(lens.mean(), color='orange', linestyle='--', label=f'Mean={lens.mean():.1f}')
        ax.legend()
    ax.set_title(f'Domain {d} — Seq Length ({rep_col})')
    ax.set_xlabel('Sequence Length')
    ax.set_ylabel('Count')
fig.suptitle('Sequence Length Distribution by Domain', fontsize=14)
fig.tight_layout()
plt.savefig('eda/figures/eda_seq_len_dist.png', dpi=150)
plt.close()

In [20]:
# Per-domain: aggregate vocabulary analysis
print("=== Sequence Domain Vocabulary Analysis ===")
for d, cols in sorted(seq_domains.items()):
    # Collect all non-null values across all features in this domain
    all_vals = []
    for c in cols:
        non_null = df[c].dropna()
        for lst in non_null:
            all_vals.extend(lst)
    all_vals = np.array(all_vals)
    print(f"\nDomain {d}:")
    print(f"  Total elements: {len(all_vals)}")
    print(f"  Unique values: {len(set(all_vals))}")
    print(f"  Value range: [{all_vals.min()}, {all_vals.max()}]")
    # Top-10 most common values
    from collections import Counter
    top10 = Counter(all_vals).most_common(10)
    print(f"  Top-10 values: {top10}")

=== Sequence Domain Vocabulary Analysis ===

Domain a:
  Total elements: 6309774


  Unique values: 345161
  Value range: [-1, 1772725487]


  Top-10 values: [(np.int64(0), 2390326), (np.int64(2), 552709), (np.int64(6), 533221), (np.int64(1), 237890), (np.int64(149), 176555), (np.int64(840), 129054), (np.int64(14), 114015), (np.int64(11), 105770), (np.int64(16), 90833), (np.int64(7), 83216)]



Domain b:
  Total elements: 7990612
  Unique values: 656112
  Value range: [-1, 1772725642]


  Top-10 values: [(np.int64(0), 3239728), (np.int64(19), 176992), (np.int64(15), 161069), (np.int64(6), 135136), (np.int64(22), 90547), (np.int64(26), 87790), (np.int64(24), 78400), (np.int64(-1), 75424), (np.int64(110), 72983), (np.int64(628), 67007)]



Domain c:
  Total elements: 5393172
  Unique values: 976068
  Value range: [-1, 1772725680]


  Top-10 values: [(np.int64(0), 591698), (np.int64(1), 435218), (np.int64(31), 336277), (np.int64(2), 333764), (np.int64(789853), 134115), (np.int64(956826), 77635), (np.int64(5), 76273), (np.int64(10), 46655), (np.int64(16797), 44509), (np.int64(3), 42416)]



Domain d:
  Total elements: 10998590
  Unique values: 192970
  Value range: [0, 1772725620]


  Top-10 values: [(np.int64(0), 3132689), (np.int64(3), 1080433), (np.int64(2), 422810), (np.int64(548), 329009), (np.int64(13), 152193), (np.int64(10), 131242), (np.int64(11), 129628), (np.int64(511), 104088), (np.int64(14), 88710), (np.int64(67), 77791)]


## 8. Feature Completeness / Missingness

In [21]:
# Missingness heatmap for scalar features
scalar_cols = []
for c in df.columns:
    if c in meta_cols:
        continue
    if get_col_type(df[c]) == 'scalar':
        scalar_cols.append(c)

# Create a binary missingness matrix for a sample (to keep the plot readable)
sample_size = min(200, len(df))
sample_df = df[scalar_cols].sample(sample_size, random_state=42)
missing_matrix = sample_df.isna().astype(int)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(missing_matrix.T, cbar=False, cmap='Blues', ax=ax,
            xticklabels=False, yticklabels=True)
ax.set_title(f'Missingness Pattern (Sample: {sample_size} rows, {len(scalar_cols)} scalar features)')
ax.set_xlabel('Sample Rows')
ax.set_ylabel('Features')
fig.tight_layout()
plt.savefig('eda/figures/eda_missingness.png', dpi=150)
plt.close()

In [22]:
# Overall missingness summary by feature group
print("=== Missingness Summary by Group ===")
for group_name, cols in col_groups.items():
    if not cols:
        continue
    scalar_c = [c for c in cols if get_col_type(df[c]) == 'scalar']
    list_c = [c for c in cols if get_col_type(df[c]) == 'list']
    if scalar_c:
        null_pct = df[scalar_c].isna().mean().mean() * 100
        print(f"  {group_name:15s} (scalar): avg null rate = {null_pct:.1f}%")
    if list_c:
        null_pct = df[list_c].isna().mean().mean() * 100
        print(f"  {group_name:15s} (list):   avg null rate = {null_pct:.1f}%")

=== Missingness Summary by Group ===
  Meta            (scalar): avg null rate = 0.0%


  User Int        (scalar): avg null rate = 31.7%
  User Int        (list):   avg null rate = 17.3%
  Item Int        (scalar): avg null rate = 19.4%
  Item Int        (list):   avg null rate = 43.9%
  User Dense      (list):   avg null rate = 9.9%
  Sequences       (list):   avg null rate = 2.3%


In [23]:
# Missingness vs label correlation
print("\n=== Missingness by Label ===")
df['is_positive'] = (df['label_type'] == 2).astype(int)
for group_name, cols in col_groups.items():
    if not cols:
        continue
    scalar_c = [c for c in cols if get_col_type(df[c]) == 'scalar']
    if not scalar_c:
        continue
    for c in scalar_c:
        missing_pct = df[c].isna().mean() * 100
        if missing_pct > 5:  # Only show features with >5% missing
            missing_pos = df[df['is_positive'] == 1][c].isna().mean() * 100
            missing_neg = df[df['is_positive'] == 0][c].isna().mean() * 100
            fid = c.split('_')[-1]
            print(f"  {c:30s}: overall={missing_pct:.1f}%, pos={missing_pos:.1f}%, neg={missing_neg:.1f}%")


=== Missingness by Label ===
  user_int_feats_54             : overall=36.8%, pos=37.1%, neg=36.8%
  user_int_feats_58             : overall=15.0%, pos=14.5%, neg=15.1%
  user_int_feats_59             : overall=15.0%, pos=14.5%, neg=15.1%
  user_int_feats_82             : overall=20.4%, pos=23.4%, neg=20.0%
  user_int_feats_86             : overall=69.2%, pos=72.6%, neg=68.7%
  user_int_feats_92             : overall=49.4%, pos=54.8%, neg=48.6%
  user_int_feats_93             : overall=17.1%, pos=19.4%, neg=16.8%
  user_int_feats_94             : overall=52.1%, pos=53.2%, neg=51.9%
  user_int_feats_95             : overall=31.8%, pos=34.7%, neg=31.4%
  user_int_feats_96             : overall=67.8%, pos=74.2%, neg=66.9%
  user_int_feats_97             : overall=29.2%, pos=25.0%, neg=29.8%
  user_int_feats_98             : overall=10.3%, pos=8.9%, neg=10.5%
  user_int_feats_99             : overall=81.2%, pos=84.7%, neg=80.7%
  user_int_feats_100            : overall=84.5%, pos=80.6%, n

## 9. Cross-Feature Analysis

In [24]:
# User-Item co-occurrence: do certain user_id + item_id pairs appear multiple times?
pair_counts = df.groupby(['user_id', 'item_id']).size()
repeat_pairs = (pair_counts > 1).sum()
print(f"Total unique (user_id, item_id) pairs: {len(pair_counts)}")
print(f"Pairs appearing more than once: {repeat_pairs}")

Total unique (user_id, item_id) pairs: 1000
Pairs appearing more than once: 0


In [25]:
# Positive rate by key user feature values
print("\n=== Positive Rate by Selected User Features ===")
df['label'] = (df['label_type'] == 2).astype(int)

# Low-cardinality scalar user features
low_card_ui = []
for entry in user_int_summary:
    if entry['type'] == 'scalar' and entry.get('nunique', 0) <= 10 and entry['null_pct'] < 50:
        low_card_ui.append(entry['col'])

for c in low_card_ui:
    grouped = df.groupby(c)['label'].agg(['mean', 'count'])
    grouped.columns = ['positive_rate', 'count']
    print(f"\n{c}:")
    print(grouped.to_string())


=== Positive Rate by Selected User Features ===

user_int_feats_1:
                  positive_rate  count
user_int_feats_1                      
1                      0.122807    171
2                      0.094340     53
4                      0.126289    776

user_int_feats_49:
                   positive_rate  count
user_int_feats_49                      
1.0                     0.130120    415
2.0                     0.117647    578

user_int_feats_50:
                   positive_rate  count
user_int_feats_50                      
0.0                     0.500000      2
1.0                     0.123742    994

user_int_feats_51:
                   positive_rate  count
user_int_feats_51                      
40.0                    1.000000      1
56.0                    0.123618    995
78.0                    0.000000      1
113.0                   0.000000      1
150.0                   0.000000      1

user_int_feats_58:
                   positive_rate  count
user_int_feats_58

In [26]:
# Positive rate by sequence presence
print("\n=== Positive Rate by Sequence Presence ===")
for d, cols in sorted(seq_domains.items()):
    rep = cols[0]
    has_seq = df[rep].notna().astype(int)
    pos_with = df[has_seq == 1]['label'].mean()
    pos_without = df[has_seq == 0]['label'].mean() if (has_seq == 0).sum() > 0 else None
    print(f"  domain_{d}: has_seq positive_rate={pos_with:.4f}, "
          f"no_seq positive_rate={pos_without if pos_without is not None else 'N/A'}")


=== Positive Rate by Sequence Presence ===
  domain_a: has_seq positive_rate=0.1226, no_seq positive_rate=0.4
  domain_b: has_seq positive_rate=0.1255, no_seq positive_rate=0.0
  domain_c: has_seq positive_rate=0.1242, no_seq positive_rate=0.0
  domain_d: has_seq positive_rate=0.1283, no_seq positive_rate=0.075


In [27]:
# Positive rate by timestamp bucket
print("\n=== Positive Rate by Timestamp Bucket ===")
ts_buckets = pd.qcut(df['timestamp'], q=5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
ts_pos_rate = df.groupby(ts_buckets)['label'].agg(['mean', 'count'])
ts_pos_rate.columns = ['positive_rate', 'count']
print(ts_pos_rate.to_string())


=== Positive Rate by Timestamp Bucket ===
           positive_rate  count
timestamp                      
Q1              0.149254    201
Q2              0.100503    199
Q3              0.118812    202
Q4              0.140704    199
Q5              0.110553    199


/var/folders/6b/6zmc4mw95wz1kwsyzb3t5gqh0000gn/T/ipykernel_48393/2766674777.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ts_pos_rate = df.groupby(ts_buckets)['label'].agg(['mean', 'count'])


## 10. Summary & Insights

In [28]:
print("=" * 70)
print("TAAC2026 Dataset EDA Summary")
print("=" * 70)

print(f"\n1. DATA SIZE: {len(df)} samples, {len(df.columns)} features")
print(f"   Time window: {duration_sec}s ({duration_sec/60:.1f} minutes)")

print(f"\n2. LABEL: Binary classification (click=1 vs conversion=2)")
print(f"   Positive rate: {positive_rate*100:.2f}% ({counts[1]} conversions out of {len(df)})")
print(f"   Class imbalance: ~1:{(1-positive_rate)/positive_rate:.1f}")
print(f"   Typical conversion delay: median={delta.median():.0f}s, mean={delta.mean():.0f}s")

print(f"\n3. USER FEATURES:")
print(f"   {len(user_int_cols)} integer features (scalar + list/multi-hot)")
print(f"   {len(user_dense_cols)} dense/vector features")
n_user_scalar = sum(1 for e in user_int_summary if e['type'] == 'scalar')
n_user_list = sum(1 for e in user_int_summary if e['type'] == 'list')
print(f"     -> {n_user_scalar} scalar, {n_user_list} list/multi-hot")
high_null_ui = [e for e in user_int_summary if e['null_pct'] > 50]
if high_null_ui:
    print(f"   High missingness (>50%): {', '.join(e['col'] for e in high_null_ui)}")

print(f"\n4. ITEM FEATURES:")
print(f"   {len(item_int_cols)} integer features")
print(f"   {len(item_dense_cols)} dense features (none in this dataset)")
n_item_scalar = sum(1 for e in item_int_summary if e['type'] == 'scalar')
n_item_list = sum(1 for e in item_int_summary if e['type'] == 'list')
print(f"     -> {n_item_scalar} scalar, {n_item_list} list/multi-hot")
high_null_ii = [e for e in item_int_summary if e['null_pct'] > 50]
if high_null_ii:
    print(f"   High missingness (>50%): {', '.join(e['col'] for e in high_null_ii)}")

print(f"\n5. SEQUENCE FEATURES:")
print(f"   4 domains (a, b, c, d) with {len(seq_cols)} total features")
for d, cols in sorted(seq_domains.items()):
    domain_data = seq_df[seq_df['domain'] == d]
    avg_null = domain_data['null_pct'].mean()
    avg_len = domain_data['len_mean'].mean() if 'len_mean' in domain_data.columns else 0
    print(f"   domain_{d}: {len(cols)} features, avg null={avg_null:.1f}%, avg len={avg_len:.0f}")

print(f"\n6. DATA QUALITY NOTES:")
print(f"   - No item_dense_feats columns present")
print(f"   - Several user features have >50% missing values (likely sparse signals)")
print(f"   - domain_d has highest sequence null rate (~8%)")
print(f"   - user_int_feats_99/100/101/102/103/109 have >80% missing")
print(f"   - Sequence timestamps can be used for time-bucket encoding (dataset.py: BUCKET_BOUNDARIES)")

print(f"\n7. FEATURE ENGINEERING SUGGESTIONS:")
print(f"   - Handle missing values: impute or use indicator flags for high-null features")
print(f"   - Use time-bucket embeddings for sequence temporal patterns")
print(f"   - Group NS features per ns_groups.json for tokenization")
print(f"   - Consider user/item frequency features (exposure counts)")
print(f"   - Conversion delay may be useful as an auxiliary signal")
print(f"   - High-cardinality features may benefit from embedding cold-start strategies")

TAAC2026 Dataset EDA Summary

1. DATA SIZE: 1000 samples, 122 features
   Time window: 781s (13.0 minutes)

2. LABEL: Binary classification (click=1 vs conversion=2)
   Positive rate: 12.40% (124 conversions out of 1000)
   Class imbalance: ~1:7.1
   Typical conversion delay: median=192s, mean=228s

3. USER FEATURES:
   46 integer features (scalar + list/multi-hot)
   10 dense/vector features
     -> 35 scalar, 11 list/multi-hot
   High missingness (>50%): user_int_feats_60, user_int_feats_86, user_int_feats_94, user_int_feats_96, user_int_feats_99, user_int_feats_100, user_int_feats_101, user_int_feats_102, user_int_feats_103, user_int_feats_108, user_int_feats_109

4. ITEM FEATURES:
   14 integer features
   0 dense features (none in this dataset)
     -> 13 scalar, 1 list/multi-hot
   High missingness (>50%): item_int_feats_83, item_int_feats_84, item_int_feats_85

5. SEQUENCE FEATURES:
   4 domains (a, b, c, d) with 45 total features
   domain_a: 9 features, avg null=0.5%, avg len=